# **Parallel chains**

In [ ]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda


# ============================================================
# LLM
# ============================================================

llm_openai = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)


# ============================================================
# TASK 1 - PROMPT
# ============================================================

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief : {input}")
])


# ============================================================
# TASK 2 - LLM
# ============================================================

llm_openai = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)


# ============================================================
# TASK 3 - STRING OUTPUT PARSER
# ============================================================

str_parser = StrOutputParser()


# ============================================================
# TASK 4 - CUSTOM RUNNABLE
# ============================================================

def dictionary_maker(text: str) -> dict:
    return {"text": text}


dictionary_maker_runnable = RunnableLambda(dictionary_maker)


# ============================================================
# PARALLEL CHAIN 1 - LINKEDIN
# ============================================================

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    (
        "human",
        "Create a post for the following text for LinkedIn: {text}"
    )
])

llm_openai = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)

str_parser = StrOutputParser()

chain_linkedin = (
    linkedin_prompt
    | llm_openai
    | str_parser
)


# ============================================================
# PARALLEL CHAIN 2 - INSTAGRAM
# ============================================================

def insta_chain(text: dict):

    text = text["text"]

    # Prompt
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a LinkedIn post generator"),
        (
            "human",
            "Create a post for the following text for Instagram: {text}"
        )
    ])

    # LLM
    llm_openai = ChatOpenAI(
        model="gpt-5-mini",
        temperature=0
    )

    # String parser
    str_parser = StrOutputParser()

    # Instagram chain
    chain_insta = (
        insta_prompt
        | llm_openai
        | str_parser
    )

    result = chain_insta.invoke(text)

    return result


insta_chain_runnable = RunnableLambda(insta_chain)


# ============================================================
# FINAL ORCHESTRATION
# ============================================================

final_chain = (
    prompt_template
    | llm_openai
    | str_parser
    | dictionary_maker_runnable
    | RunnableParallel(
        branches={
            "linkedin": chain_linkedin,
            "instagram": insta_chain_runnable
        }
    )
)


# ============================================================
# INVOKE FINAL CHAIN
# ============================================================

result = final_chain.invoke("KGF")

print(result)


# ============================================================
# BEAUTIFY FUNCTION
# ============================================================

def beautify(final_response: dict) -> dict:

    linkedin_response = final_response["branches"]["linkedin"]
    instagram_response = final_response["branches"]["instagram"]

    return {
        "linkedin": linkedin_response,
        "instagram": instagram_response
    }


beautify_runnable = RunnableLambda(beautify)


# ============================================================
# BEAUTIFIED CHAIN
# ============================================================

beautified_chain = final_chain | beautify_runnable


# ============================================================
# FINAL INVOCATION
# ============================================================

result = beautified_chain.invoke("Pushpa")

print(result)